In [1]:
import numpy as np
from deepcave import Recorder, Objective
from deepcave.runs.converters.deepcave import DeepCAVERun
from deepcave.plugins.summary.footprint import FootPrint
from deepcave.plugins.hyperparameter.importances import Importances
import io 
from PIL import Image
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as plt
from arlbench.core.algorithms import DQN, PPO, SAC
from pathlib import Path
from ConfigSpace import ConfigurationSpace, Float, Categorical
import math
import os

deepcave.utils.styled_plot (WARNING): LaTeX not found. Using default font.


In [2]:
replacement_dict = {"atari_qbert": "atari", 'atari_double_dunk': "atari", 'atari_phoenix': "atari", 'atari_this_game': "atari", 
                    'box2d_lunar_lander': "box2d", "box2d_bipedal_walker": "box2d", 'cc_acrobot': "cc", 'cc_cartpole': "cc", 'cc_mountain_car': "cc", "cc_pendulum": "cc", 'cc_continuous_mountain_car': "cc",
                    'minigrid_door_key': "minigrid", 'minigrid_empty_random': "minigrid", 'minigrid_four_rooms': "minigrid", 
                    'minigrid_unlock': "minigrid", "brax_ant": "brax", "brax_halfcheetah": "brax", "brax_hopper": "brax", "brax_humanoid": "brax"}
algorithms = {"dqn": DQN, "ppo": PPO, "sac": SAC}
envs = ["atari_qbert", 'atari_double_dunk', 'atari_phoenix', 'atari_this_game', 
        'box2d_lunar_lander', 'box2d_bipedal_walker', 'cc_acrobot', 'cc_cartpole', 'cc_mountain_car', 'cc_pendulum', 'cc_continuous_mountain_car',
        'minigrid_door_key', 'minigrid_empty_random', 'minigrid_four_rooms', 'minigrid_unlock', 'brax_ant', 'brax_halfcheetah', 'brax_hopper', 'brax_humanoid']


In [3]:
def plotly_fig2array(fig):
    #convert Plotly fig to  an array
    fig_bytes = fig.to_image(format="png")
    buf = io.BytesIO(fig_bytes)
    img = Image.open(buf)
    return np.asarray(img)

In [41]:
def data_to_deepcave(algorithm, domain=False, save_path=None):
    target_path = f"../results_combined/sobol/{domain}_{algorithm}.csv"
    try:
        all_configs = pd.read_csv(target_path)
    except:
        print(f"Could not read {target_path}")
        return

    default_configspace = algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names()
    configspace = ConfigurationSpace()
    for c in default_configspace:
        if c in ["buffer_prio_sampling", "use_target_network", "alpha_auto", "normalize_advantage"]:
            hp = Categorical(c, [True, False])
            configspace.add_hyperparameter(hp)
        else:
            key = [a for a in all_configs.keys() if c in a]
            if len(key) > 0:
                key = key[0]
                hp_min = all_configs[key].min()
                hp_max = all_configs[key].max()
                bounds = (min(0, hp_min-0.1*hp_min), hp_max+0.1*hp_max)
                hp = Float(c, bounds=bounds)
                configspace.add_hyperparameter(hp)
    default = configspace.get_default_configuration()
    accuracy = Objective("reward_mean", lower=min(all_configs["performance"]), upper=max(all_configs["performance"]), optimize="upper")
    save_path = save_path if save_path else f"deepcave_logs/{algorithm}_{domain}"

    with Recorder(configspace, objectives=[accuracy], save_path=save_path) as r:
        for index, d in all_configs.iterrows():
            current_hps = d
            config = default
            for c in current_hps.keys():
                key = c.split(".")[-1]
                if key in config.keys():
                    if math.isnan(current_hps[c]):
                        current_hps[c] = 0
                    try:
                        value = float(current_hps[c])
                    except:
                        current_hps[c]
                    config[key] = value
            r.start(config, 1)
            r.end(costs=current_hps["performance"], seed=d["seed"])
    return save_path

In [50]:
def get_importances(path, algorithm, method="local"):
    # Instantiate the run
    run = DeepCAVERun.from_path(Path(path))
    objective_id = run.get_objective_ids()[0]
    budget_ids = run.get_budget_ids()

    # Instantiate the plugin
    plugin = Importances()

    inputs = plugin.generate_inputs(
        objective_id=objective_id,
        hyperparameter_names= algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names(), 
        budget_ids=budget_ids,
        method=method, n_hps=10, n_trees=10
    )
    # Note: Filter variables are not considered.
    outputs = plugin.generate_outputs(run, inputs)

    # Finally, you can load the figure. Here, the filter variables play a role.
    # Alternatively: Use the matplotlib output (`load_mpl_outputs`) if available.
    figure = plugin.load_outputs(run, inputs, outputs)  # plotly.go figure
    return figure

In [5]:
def get_footprint(path):
    # Instantiate the run
    run = DeepCAVERun.from_path(Path(path))
    objective_id = run.get_objective_ids()[0]
    budget_id = run.get_budget_ids()[-1]

    # Instantiate the plugin
    plugin = FootPrint()

    inputs = plugin.generate_inputs(
        objective_id=objective_id,
        budget_id=budget_id,
        details=True,
        show_supports=True,
        show_borders=True,
    )
    # Note: Filter variables are not considered.
    outputs = plugin.generate_outputs(run, inputs)

    # Finally, you can load the figure. Here, the filter variables play a role.
    # Alternatively: Use the matplotlib output (`load_mpl_outputs`) if available.
    figure = plugin.load_outputs(run, inputs, outputs)  # plotly.go figure
    return figure

In [6]:
def normalize_env_scores(data):
    groups = data.groupby("Environment")["Score"]
    data["Score"] = (data["Score"] - groups.transform('min')) / (groups.transform('max') - groups.transform('min'))
    return data

In [8]:
def get_boxenplot(data, domain=False, domain_single=None):
    if domain:
        data["Environment"] = data["Environment"].replace(replacement_dict)
    if domain_single in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain_single]
        data = data[data["Environment"].isin(subdomains)]
    fig = sns.boxenplot(data, x="Environment", y="Score")
    plt.xticks(rotation=70)
    plt.tight_layout()
    return fig

def get_kdeplot(data, domain=False, domain_single=None):
    if domain:
        data["Environment"] = data["Environment"].replace(replacement_dict)
    if domain_single in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain_single]
        data = data[data["Environment"].isin(subdomains)]
    return sns.kdeplot(data, x="Score", hue="Environment")

def get_pairplot(data, domain=False, domain_single=None):
    if domain:
        data["Environment"] = data["Environment"].replace(replacement_dict)
    if domain_single in ["atari", "cc", "box2d", "minigrid"]:
        subdomains = [k for k in replacement_dict.keys() if replacement_dict[k] == domain_single]
        data = data[data["Environment"].isin(subdomains)]
    return sns.pairplot(data, hue="Environment", kind="kde")

In [11]:
def load_data(algorithm):
    source_file = f"results/{algorithm}_4_.csv"
    #meta_file = f"data/{algorithm}/metadata.json"
    #with open(meta_file) as f:
    #    metadata = json.load(f)
    return pd.read_csv(source_file)#, metadata

In [ ]:
algorithm = "sac"
#performance_data = load_data(algorithm)
#performance_data = normalize_env_scores(performance_data)
deepcave_paths = []
for env in envs:
   if not os.path.isdir(f"deepcave_logs/{algorithm}_{env}"):
       print(f"Parsing {env}")
       data_path = data_to_deepcave(algorithm, domain=env)
       deepcave_paths.append(data_path)

Parsing atari_qbert
Parsing atari_double_dunk
Parsing atari_phoenix
Parsing atari_this_game
Parsing box2d_lunar_lander
Parsing box2d_bipedal_walker
Could not read ../results_combined/sobol/box2d_bipedal_walker_ppo.csv
Parsing cc_acrobot
Parsing cc_cartpole
Parsing cc_mountain_car
Parsing cc_pendulum
Parsing cc_continuous_mountain_car
Parsing minigrid_door_key
Parsing minigrid_empty_random
Parsing minigrid_four_rooms
Parsing minigrid_unlock
Parsing brax_ant
Parsing brax_halfcheetah
Parsing brax_hopper
Parsing brax_humanoid


In [25]:
get_boxenplot(performance_data.copy(), True)
plt.figure()
get_boxenplot(performance_data.copy())
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="atari")
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="cc")
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="minigrid")
plt.figure()
get_boxenplot(performance_data.copy(), domain_single="box2d")


<Axes: xlabel='Environment', ylabel='Score'>

In [26]:
get_kdeplot(performance_data.copy(), True)
plt.figure()
get_kdeplot(performance_data.copy())
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="atari")
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="cc")
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="minigrid")
plt.figure()
get_kdeplot(performance_data.copy(), domain_single="box2d")

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_23465/3652807460.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()


<Axes: xlabel='Score', ylabel='Density'>

In [27]:
get_pairplot(performance_data.copy(), True)
plt.figure()
get_pairplot(performance_data.copy())
plt.figure()
get_pairplot(performance_data.copy(), domain_single="atari")
plt.figure()
get_pairplot(performance_data.copy(), domain_single="cc")
plt.figure()
get_pairplot(performance_data.copy(), domain_single="minigrid")
plt.figure()
get_pairplot(performance_data.copy(), domain_single="box2d")

In [54]:
for data_path in deepcave_paths:
    print(data_path)
    try:
        img = get_footprint(Path(data_path) / "run_1")
        img[0].write_image(data_path + "/footprint.png")
    except Exception as e:
        print("Error")
        print(e)

deepcave_logs/ppo_atari_qbert


100%|██████████| 1280/1280 [00:03<00:00, 336.55it/s]

deepcave.evaluators.footprint (INFO): Added 1280 configurations.
deepcave.evaluators.footprint (INFO): Starting to calculate distances and add border and random configurations...


deepcave.evaluators.footprint (INFO): Found 1300 configurations...
deepcave.evaluators.footprint (INFO): Found 1400 configurations...
deepcave.evaluators.footprint (INFO): Found 1500 configurations...
deepcave.evaluators.footprint (INFO): Found 1600 configurations...
deepcave.evaluators.footprint (INFO): Found 1800 configurations...
deepcave.evaluators.footprint (INFO): Found 2000 configurations...
deepcave.evaluators.footprint (INFO): Found 2100 configurations...
deepcave.evaluators.footprint (INFO): Found 2300 configurations...
deepcave.evaluators.footprint (INFO): Found 2400 configurations...
deepcave.evaluators.footprint (INFO): Found 2500 configurations...
deepcave.evaluators.footprint (INFO): Found 2600 configurations...
deepcave.evaluators.footprint (INFO): Found 2800 configurations...
deepcave.evaluators.footprint (INFO): Found 2900 configurations...
deepcave.evaluators.footprint (INFO): Found 3000 configurations...
deepcave.evaluators.footprint (INFO): Found 3100 configuration

100%|██████████| 1280/1280 [00:03<00:00, 322.94it/s]

...

deepcave.evaluators.footprint (INFO): Added 2560 configurations.
deepcave.evaluators.footprint (INFO): Starting to calculate distances and add border and random configurations...


deepcave.evaluators.footprint (INFO): Found 2600 configurations...
deepcave.evaluators.footprint (INFO): Found 2800 configurations...
deepcave.evaluators.footprint (INFO): Found 2900 configurations...
deepcave.evaluators.footprint (INFO): Found 3000 configurations...
deepcave.evaluators.footprint (INFO): Found 3200 configurations...
deepcave.evaluators.footprint (INFO): Found 3300 configurations...
deepcave.evaluators.footprint (INFO): Found 3500 configurations...
deepcave.evaluators.footprint (INFO): Found 3600 configurations...
deepcave.evaluators.footprint (INFO): Found 3700 configurations...
deepcave.evaluators.footprint (INFO): Found 3900 configurations...
deepcave.evaluators.footprint (INFO): Found 4000 configurations...
deepcave.evaluators.footprint (INFO): Added 438 border configs and 1003 random configs.
deepcave.evaluators.footprint (INFO): Total configurations: 4001.
deepcave.evaluators.footprint (INFO): Getting MDS data...
deepcave.evaluators.footprint (INFO): Training on o

100%|██████████| 2560/2560 [00:15<00:00, 161.84it/s]

deepcave.evaluators.footprint (INFO): Added 2560 configurations.
deepcave.evaluators.footprint (INFO): Starting to calculate distances and add border and random configurations...


deepcave.evaluators.footprint (INFO): Found 2600 configurations...
deepcave.evaluators.footprint (INFO): Found 2800 configurations...
deepcave.evaluators.footprint (INFO): Found 2900 configurations...
deepcave.evaluators.footprint (INFO): Found 3000 configurations...
deepcave.evaluators.footprint (INFO): Found 3200 configurations...
deepcave.evaluators.footprint (INFO): Found 3300 configurations...
deepcave.evaluators.footprint (INFO): Found 3500 configurations...
deepcave.evaluators.footprint (INFO): Found 3600 configurations...
deepcave.evaluators.footprint (INFO): Found 3700 configurations...
deepcave.evaluators.footprint (INFO): Found 3900 configurations...
deepcave.evaluators.footprint (INFO): Found 4000 configurations...
deepcave.evaluators.footprint (INFO): Added 438 border configs and 1003 random configs.
deepcave.evaluators.footprint (INFO): Total configurations: 4001.
deepcave.evaluators.footprint (INFO): Getting MDS data...


In [ ]:
for data_path in deepcave_paths:
    print(data_path)
    try:
        img = get_importances(Path(data_path) / "run_1", algorithm)
        img.write_image(data_path + "/importances.png")
    except Exception as e:
        print("Error")
        print(e)

deepcave_logs/ppo_atari_qbert
14732.14843 17565.625
16621.972658 17565.625
17470.625 17565.625
16848.847654 17565.625
17244.746091 17565.625
14564.74608 17565.625
16417.46094 17565.625
17390.488281 17565.625
17237.265622 17565.625
2734.6288999999997 17565.625
17354.6875 17565.625
17359.335937 17565.625
17450.17578 17565.625
17243.632815 17565.625
16708.124999 17565.625
16825.449224 17565.625
17468.125 17565.625
17408.339842 17565.625
17458.125 17565.625
17480.625 17565.625
16878.847655 17565.625
16847.656252 17565.625
17033.398437 17565.625
17473.125 17565.625
17435.625 17565.625
17525.625 17565.625
13911.38671 17565.625
16208.691415 17565.625
17325.507814 17565.625
15856.445313 17565.625
17565.625 17565.625
14550.585961 17565.625
16942.812505 17565.625
17483.125 17565.625
17322.851563 17565.625
17430.625 17565.625
17442.539062 17565.625
16781.621101 17565.625
17226.269531 17565.625
13454.21873 17565.625
14446.19138 17565.625
17346.1914055 17565.625
14565.878904000001 17565.625
17333.0

In [ ]:
for data_path in deepcave_paths:
    print(data_path)
    try:
        img = get_importances(Path(data_path) / "run_1", algorithm, method="global")
        img.write_image(data_path + "/importances_global.png")
    except Exception as e:
        print("Error")
        print(e)

deepcave_logs/ppo_atari_qbert
15004.2969 17565.625
16184.9609 17565.625
13987.1094 17565.625
15990.2344 17565.625
14634.375 17565.625
14725.7812 17565.625
14050.3906 17565.625
14370.7031 17565.625
13483.9844 17565.625
14889.6484 17565.625
15849.8047 17565.625
16070.3125 17565.625
16798.4375 17565.625
16815.625 17565.625
16768.75 17565.625
16893.3594 17565.625
16752.14844 17565.625
16828.5156 17565.625
16676.5625 17565.625
16766.21094 17565.625
17440.625 17565.625
17440.625 17565.625
17565.625 17565.625
17440.625 17565.625
17415.625 17565.625
17565.625 17565.625
17415.625 17565.625
17415.625 17565.625
17440.625 17565.625
17565.625 17565.625
16859.5703 17565.625
17032.03125 17565.625
16763.28125 17565.625
16928.125 17565.625
16836.52344 17565.625
16898.2422 17565.625
16798.6328 17565.625
16865.625 17565.625
16642.1875 17565.625
16864.2578 17565.625
17372.65625 17565.625
17318.75 17565.625
17330.07812 17565.625
17060.9375 17565.625
17047.65625 17565.625
17311.91406 17565.625
17003.3203 17